# Data Loading

Loading the **Online Retail II** dataset (UCI ML Repository, id 502) — a UK-based online retailer's transactions from Dec 2009 to Dec 2011. The raw file has two sheets (one per year), so we load and combine both.

In [2]:
import pandas as pd
df_2009_2010 =  pd.read_excel("online_retail_II.xlsx", sheet_name="Year 2009-2010")
df_2010_2011 = pd.read_excel("online_retail_II.xlsx", sheet_name="Year 2010-2011")

df = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
print(df.shape)
df.info()

(1067371, 8)
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


# Data Cleaning

Removing rows that don't represent valid product sales: missing customer IDs, cancelled orders, and invalid quantity/price values. Also creating a `TotalPrice` column for revenue calculations.

In [4]:
df = df.dropna(subset=["Customer ID"])

df = df[~df['Invoice'].astype(str).str.startswith("C")]

df= df[(df["Quantity"]> 0) & (df["Price"]> 0)]

df["TotalPrice"] = df['Quantity'] * df['Price']

df.shape

(805549, 9)

## Checking for Duplicates and Date Range

Sanity-checking the cleaned data before moving further — looking for exact duplicate rows and confirming the date range matches expectations.

In [5]:
print("Duplicate rows:", df.duplicated().sum())

print("Date range:", df['InvoiceDate'].min(), "to", df["InvoiceDate"].max())

df[["Quantity", "Price", "TotalPrice"]].describe()

Duplicate rows: 26124
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


,Quantity,Price,TotalPrice
count,805549.000000,805549.000000,805549.000000
mean,13.290522,3.206561,22.026505
std,143.634088,29.199173,224.041928
min,1.000000,0.001000,0.001000
25%,2.000000,1.250000,4.950000
50%,5.000000,1.950000,11.850000
75%,12.000000,3.750000,19.500000
max,80995.000000,10953.500000,168469.600000


In [6]:
df= df.drop_duplicates()
df.shape

(779425, 9)

In [7]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


## Investigating Outliers

The `describe()` output above showed unusually high max values for `Price` and `Quantity`. Rather than assume these are errors, inspecting the actual rows to see what they represent.

In [8]:
df.sort_values("Price", ascending=False).head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
135013,502263,M,Manual,1,2010-03-23 15:22:00,10953.50,12918.0,United Kingdom,10953.50
358639,524159,M,Manual,1,2010-09-27 16:12:00,10468.80,14063.0,United Kingdom,10468.80
74356,496115,M,Manual,1,2010-01-29 11:04:00,8985.60,17949.0,United Kingdom,8985.60
698843,551697,POST,POSTAGE,1,2011-05-03 13:46:00,8142.75,16029.0,United Kingdom,8142.75
129987,501768,M,Manual,1,2010-03-19 11:45:00,6958.17,15760.0,Norway,6958.17
129903,501766,M,Manual,1,2010-03-19 11:35:00,6958.17,15760.0,Norway,6958.17
947835,573077,M,Manual,1,2011-10-27 14:13:00,4161.06,12536.0,France,4161.06
947838,573080,M,Manual,1,2011-10-27 14:20:00,4161.06,12536.0,France,4161.06
931867,571751,M,Manual,1,2011-10-19 11:18:00,3949.32,12744.0,Singapore,3949.32
288706,517483,M,Manual,1,2010-07-29 12:29:00,3610.50,12737.0,France,3610.50


In [9]:
df.sort_values("Quantity", ascending=False).head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,168469.6
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,77183.6
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,0.10,13902.0,Denmark,1915.2
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark,1296.0
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark,1296.0
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,2010-03-17 13:09:00,0.10,13902.0,Denmark,1274.4
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,2010-03-17 13:09:00,0.10,13902.0,Denmark,1248.0
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom,2500.0
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom,2500.0
135030,502269,21981,PACK OF 12 WOODLAND TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom,2500.0


## Identifying Non-Product Stock Codes

The price outliers turned out to be non-product entries (postage, manual adjustments) mixed into the transaction log. Finding all stock codes that are purely letters, since real product codes are numeric.

In [10]:
special_codes = df[df["StockCode"].str.match(r'^[A-Za-z]+$', na=False)]["StockCode"].unique()
print(special_codes)

###
for code in ['POST', 'M', 'PADS', 'ADJUST', 'D', 'DOT']:
    print(code, "->", df[df["StockCode"] == code]["Description"].unique())

['POST' 'M' 'PADS' 'ADJUST' 'D' 'DOT']
POST -> ['POSTAGE']
M -> ['Manual']
PADS -> ['PADS TO MATCH ALL CUSHIONS']
ADJUST -> ['Adjustment by john on 26/01/2010 16'
 'Adjustment by john on 26/01/2010 17']
D -> ['Discount']
DOT -> ['DOTCOM POSTAGE']


In [11]:
non_product_codes = ['POST', 'M', 'ADJUST', 'D', 'DOT']  # PADS excluded on purpose, it's a real product
df = df[~df["StockCode"].isin(non_product_codes)]
df = df.drop_duplicates()

df.shape

(776888, 9)

## Fixing Inconsistent Product Descriptions

The same `StockCode` can have multiple different `Description` text values (typos, naming changes over time). Standardizing each product to a single description so later analysis (like "top products by revenue") isn't split across duplicate names.

In [12]:
desc_per_code = df.groupby("StockCode")["Description"].nunique()
inconsistent = desc_per_code[desc_per_code > 1].sort_values(ascending=False)
print("StockCodes with multiple descriptions:", len(inconsistent))
inconsistent.head(10)

StockCodes with multiple descriptions: 623


StockCode
20685     4
23196     4
23236     4
22345     4
22344     4
22384     4
22346     4
21955     4
20750     3
84509C    3
Name: Description, dtype: int64

In [13]:
df["StockCode"] = df["StockCode"].astype(str)

df[df["StockCode"] == "20685"][["StockCode", "Description"]].drop_duplicates()

,StockCode,Description
524,20685,RED SPOTTY COIR DOORMAT
79362,20685,DOOR MAT RED SPOT
252992,20685,DOORMAT RED SPOT
352536,20685,DOORMAT RED RETROSPOT


In [14]:
most_common_desc = df.groupby("StockCode")["Description"].agg(lambda x: x.value_counts().index[0])
df["Description"] = df["StockCode"].map(most_common_desc)

# Confirm it worked
(df.groupby("StockCode")["Description"].nunique() > 1).sum()

np.int64(0)

## Saving the Cleaned Data

Exporting the cleaned dataset to CSV so the analysis notebook can load it directly, without re-running the full download-and-clean pipeline each time.

In [16]:
df.to_csv("cleaned_retail_data.csv", index=False)